# Normalización, patrones y práctica guiada

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/2-normalizacion-y-patrones.ipynb)

Este notebook reúne las herramientas clásicas que más adelante seguiremos necesitando: stemming como comparación histórica, lematización como estrategia principal de normalización, reglas con `Matcher` y `PhraseMatcher`, y una práctica guiada para consolidar los conceptos sobre texto real.

## Referencias
* [spaCy Rule-based Matching](https://spacy.io/usage/rule-based-matching)
* [NLTK Stemmers](https://www.nltk.org/howto/stem.html)

## Preparación del entorno
Asumiendo que la librería ya se encuentra instalada, dependiendo de la tarea, necesitamos descargar el modelo o las dependencias puntuales antes de empezar.


In [13]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

/tmp/ipykernel_1317889/2396000874.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [14]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt

In [2]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 37.8 MB/s eta 0:00:00 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## Normalización: stemming vs lematización

Antes de pasar a vocabularios y reglas, conviene distinguir dos ideas que suelen confundirse. El *stemming* recorta palabras siguiendo heurísticas; la *lematización* intenta recuperar una forma canónica usando información lingüística. En la práctica moderna solemos preferir la lematización, pero comparar ambos enfoques ayuda a entender sus compromisos.

## Porter Stemmer
Este es uno de los algoritmos de stemming más comunes. Desarrollado por [Martin Porter](https://en.wikipedia.org/wiki/Martin_Porter)

Observemos por ejemplo este conjunto de palabras:

In [4]:
import nltk
from nltk.stem.porter import PorterStemmer

p_stemmer = PorterStemmer()
words = ['run', 'runner', 'ran', 'runs', 'easily', 'fairly']

Ahora, si usamos el algoritmo de porter, podrémos obtener los podemos obtener una visualización inicial de posibles raices:

In [5]:
for word in words:
    print(f"{word:{20}}--> {p_stemmer.stem(word)}")

run                 --> run
runner              --> runner
ran                 --> ran
runs                --> run
easily              --> easili
fairly              --> fairli


Podemos observar que `run` y `runs` comparten la misma raíz `run`.

Hay algo a tener en cuenta: El porter stemmer original fue desarrollado para el idioma inglés ya que fue implementado siguiendo ciertas reglas consistentes de dicho idioma, por lo que no podríamos aplicarlo tal cual al español.

Sin embargo, a partir de este stemmer, el método Snowball fue desarrollado para muchos otros lenguajes.

## Snowball
Snowball es entonces una mejora sobre el porter stemmer, incluyendo soporte para otros lenguajes. Por motivos de completitud y comparación, continuemos con inglés.

In [6]:
from nltk.stem.snowball import SnowballStemmer

s_stemmer = SnowballStemmer(language='english')
for word in words:
     print(f"{word:{20}}--> {s_stemmer.stem(word)}")

run                 --> run
runner              --> runner
ran                 --> ran
runs                --> run
easily              --> easili
fairly              --> fair


Observemos la diferencia en la palabra `fair` que es mejor y mucho más general, realmente las palabras `fair` y `fairly` son ambas válidas y comparten la misma raíz.

Ahora observemos como se comporta con un conjunto de palabras un poco más difícil.

In [8]:
from typing import List

def stem(words :List[str], stemmer) -> List[str]:
    return [stemmer.stem(word) for word in words]

In [9]:
import pandas as pd

# Notice these words share roots but mean totally different things
words = ['generous', 'generation', 'generously', 'generate']
with_porter = stem(words, p_stemmer)
with_snowball = stem(words, s_stemmer)

word_list = list(zip(words, with_porter, with_snowball))
words_df = pd.DataFrame(word_list, columns=['word', 'porter', 'snowball'])
words_df

,word,porter,snowball
0,generous,gener,generous
1,generation,gener,generat
2,generously,gener,generous
3,generate,gener,generat


Aquí podemos observar como snowball es mejor que porter.

## Lematización

Ahora repetimos el ejercicio con un pipeline lingüístico. Aquí ya no nos interesa solo recortar caracteres, sino aprovechar el POS tag para obtener una forma base más útil para análisis y modelado.

Analicemos la siguiente oración:

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("I am a runner running in a race because I love to run since I ran today")

In [4]:
for token in doc:
    print(f"{token.text:{20}}{token.pos_:{20}}{token.lemma:<{20}}\t{token.lemma_:{30}}")

I                   PRON                4690420944186131903 	I                             
am                  AUX                 10382539506755952630	be                            
a                   DET                 11901859001352538922	a                             
runner              NOUN                12640964157389618806	runner                        
running             VERB                12767647472892411841	run                           
in                  ADP                 3002984154512732771 	in                            
a                   DET                 11901859001352538922	a                             
race                NOUN                8048469955494714898 	race                          
because             SCONJ               16950148841647037698	because                       
I                   PRON                4690420944186131903 	I                             
love                VERB                3702023516439754181 	love               

Observamos que para cada token obtenemos el POS (Part of Speech) y su correspondiente lemma. Nótese que los verbos son correctamente lemmatizados a su raíz.

## Patrones con vocabularios y frases

Una vez normalizamos texto, podemos construir reglas explícitas. Esto es especialmente útil cuando queremos detectar expresiones de interés, bootstrappear etiquetas o crear features para modelos más adelante.

## Matchers
Podemos pensar en los matchers de forma similar a como usamos las expresiones regulares en programación tradicional, solo que aplicado a documentos.

In [3]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab) # work with the normal vocabulary

In [4]:
# SolarPower
pattern1 = [{'LOWER': 'solarpower'}]
# Solar-power
pattern2 = [{'LOWER': 'solar'}, {'IS_PUNCT': True}, {'LOWER': 'power'}]
# Solar power
pattern3 = [{'LOWER': 'solar'}, {'LOWER': 'power'}]

Observemos el ejemplo anterior y los correspondientes patrones que queremos encontrar. Nótese que cada parte puede ser representada como un diccionario. Esto también es una forma de hacer *tagging* de los tokens, más allá de los estándar POS, que pueden ser útiles dependiendo de la tarea en cuestión.

## Otros atributos para los tokens
A parte de los lemmas, hay otra variedad de atributos para los tokens que podemos utilizar para determinar reglas de matching:

<table><tr><th>Attribute</th><th>Description</th></tr>

<tr ><td><span >`ORTH`</span></td><td>The exact verbatim text of a token</td></tr>
<tr ><td><span >`LOWER`</span></td><td>The lowercase form of the token text</td></tr>
<tr ><td><span >`LENGTH`</span></td><td>The length of the token text</td></tr>
<tr ><td><span >`IS_ALPHA`, `IS_ASCII`, `IS_DIGIT`</span></td><td>Token text consists of alphanumeric characters, ASCII characters, digits</td></tr>
<tr ><td><span >`IS_LOWER`, `IS_UPPER`, `IS_TITLE`</span></td><td>Token text is in lowercase, uppercase, titlecase</td></tr>
<tr ><td><span >`IS_PUNCT`, `IS_SPACE`, `IS_STOP`</span></td><td>Token is punctuation, whitespace, stop word</td></tr>
<tr ><td><span >`LIKE_NUM`, `LIKE_URL`, `LIKE_EMAIL`</span></td><td>Token text resembles a number, URL, email</td></tr>
<tr ><td><span >`POS`, `TAG`, `DEP`, `LEMMA`, `SHAPE`</span></td><td>The token's simple and extended part-of-speech tag, dependency label, lemma, shape</td></tr>
<tr ><td><span >`ENT_TYPE`</span></td><td>The token's entity label</td></tr>

</table>

Ahora podemos agregar estos patrones al matcher y aplicarlos a un documento:

In [9]:
patterns = [pattern1, pattern2, pattern3]
matcher.add('SolarPower', patterns)
doc = nlp('The Solar Power industry continues to grow as solarpower increases. \
Solar-power cars are gaining popularity. ')

found_matches = matcher(doc)

In [10]:
found_matches

[(8656102463236116519, 1, 3),
 (8656102463236116519, 8, 9),
 (8656102463236116519, 11, 14)]

La lista obtenida contiene los ID de los tokens y los índices desde donde comienzan y terminan dentro del documento. Podemos imprimir esta información utilizando el vocabulario con los ID de token que hemos obtenido.

In [11]:
def print_matches(found_matches):
    for match_id, start, end in found_matches:
        string_id = nlp.vocab.strings[match_id] # get string representation
        span = doc[start:end] # get a span of this particular match
        print(match_id, string_id, start, end, span.text)

print_matches(found_matches)

8656102463236116519 SolarPower 1 3 Solar Power
8656102463236116519 SolarPower 8 9 solarpower
8656102463236116519 SolarPower 11 14 Solar-power


Aquí podemos observar diferentes matches que apuntan a la misma entrada en el vocabulario (el matcher) y el correspondiente texto al que hace referencia.

Podemos remover los matchers si ya no nos interesa trabajar con ellos.

In [12]:
matcher.remove('SolarPower') # use the name we defined when we added it

Y agregar otro conjunto de marchers

In [13]:
# solarpower SolarPower
pattern1 = [{'LOWER': 'solarpower'}]
# solar.power solar-power etc
pattern2 = [{'LOWER': 'solar'}, {'IS_PUNCT': True, 'OP': '*'}, {'LOWER': 'power'}]

patterns = [pattern1, pattern2]
matcher.add('SolarPower', patterns)

In [14]:
found_matches = matcher(doc)
print_matches(found_matches)

8656102463236116519 SolarPower 1 3 Solar Power
8656102463236116519 SolarPower 8 9 solarpower
8656102463236116519 SolarPower 11 14 Solar-power


Los cuantificadores a continuación pueden ser procesados con el operador correspondiente:
<table><tr><th>OP</th><th>Description</th></tr>

<tr ><td><span >\!</span></td><td>Negate the pattern, by requiring it to match exactly 0 times</td></tr>
<tr ><td><span >?</span></td><td>Make the pattern optional, by allowing it to match 0 or 1 times</td></tr>
<tr ><td><span >\+</span></td><td>Require the pattern to match 1 or more times</td></tr>
<tr ><td><span >\*</span></td><td>Allow the pattern to match zero or more times</td></tr>
</table>


Notemos que hemos obtenido el mismo resultado pero solo con un patrón menos, esto es gracias a la opción `OP` en el segundo patrón la cual hace que, al igual que en regex, haga match con cero o más ocurrencias.

## Matchers de frases
Como otra alternativa para la construcción de patrones, podemos crear unos con frases:

In [15]:
from spacy.matcher import PhraseMatcher

matcher = PhraseMatcher(nlp.vocab)

Empecemos a trabajar con un corpus real. Vamos a usar el extracto del articulo de *Reaganomics* de la Wikipedia:

* https://en.wikipedia.org/wiki/Reaganomics

In [19]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion1/reaganomics.txt

In [16]:
with open('./reaganomics.txt', encoding='ISO-8859-1') as file:
    doc = nlp(file.read())

Y luego vamos a buscar patrones de frases como:

In [17]:
phrase_list = ['voodoo economics', 'supply-side economics', 'trickle-down economics', 'free-market economics']
phrase_patterns = [nlp(text) for text in phrase_list]
matcher.add('EconMatcher', None, *phrase_patterns)
found_matches = matcher(doc)

In [18]:
print_matches(found_matches)

3680293220734633682 EconMatcher 41 45 supply-side economics
3680293220734633682 EconMatcher 49 53 trickle-down economics
3680293220734633682 EconMatcher 54 56 voodoo economics
3680293220734633682 EconMatcher 61 65 free-market economics
3680293220734633682 EconMatcher 673 677 supply-side economics
3680293220734633682 EconMatcher 2986 2990 trickle-down economics


Hemos encontrado una serie de matches, con sus respectivos índices de donde se encuentran dichas frases.

Esto puede ser útil a la hora de construir vocabularios, recordemos que en NLP, los vocabularios no están limitados exclusivamente a palabras sueltas, muchas veces dos palabras juntas traen un significado diferente que si estuvieran separadas y valdría la pena considerarlas como un token independiente.

## Mini práctica guiada

Cerramos con una práctica corta para reforzar lectura de documentos, conteo de tokens, segmentación por oraciones y construcción de un matcher sencillo sobre un texto narrativo.

In [3]:
# RUN THIS CELL to perform standard imports:
import spacy
nlp = spacy.load('en_core_web_sm')

**1. Creamos el documento desde el archivo `owlcreek.txt`**<br>
> Pista: Usa `with open('./owlcreek.txt') as f:`

In [16]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/Sesion1/owlcreek.txt

In [4]:
with open('./owlcreek.txt') as file:
    doc = nlp(file.read())

In [5]:
doc[:36]

AN OCCURRENCE AT OWL CREEK BRIDGE

by Ambrose Bierce

I

A man stood upon a railroad bridge in northern Alabama, looking down
into the swift water twenty feet below.  

El documento fue cargado exitosamente!

**2. Cuantos tokens hay en el archivo?**

In [6]:
len(doc)

4835

**3. Cuantas oraciones hay en el archivo?**
<br>Pista: Necesitarás una lista primero

In [7]:
sentences = list(doc.sents)
len(sentences)

204

**4. Imprime la segunda oración del documento**
<br> Pista: Los índices comienzan en 0 y el título cuenta como la primera oración.

In [8]:
sentences[1]

The man's hands were behind
his back, the wrists bound with a cord.  

**5. Por cada token en la oración anterior, imprime su `text`, `POS` tag, `dep` tag y `lemma`**
<br>

In [9]:
print("{:20}{:20}{:20}{:20}".format("Text", "POS", "dep", "lemma"))
for token in sentences[1]:
    print(f"{token.text:{20}}{token.pos_:{20}}{token.dep_:{20}}{token.lemma_:{20}}")

Text                POS                 dep                 lemma               
The                 DET                 det                 the                 
man                 NOUN                poss                man                 
's                  PART                case                's                  
hands               NOUN                nsubj               hand                
were                AUX                 ROOT                be                  
behind              ADP                 prep                behind              

                   SPACE               dep                 
                   
his                 PRON                poss                his                 
back                NOUN                pobj                back                
,                   PUNCT               punct               ,                   
the                 DET                 det                 the                 
wrists              NOUN    

**6. Implementa un matcher llamado *Swimming* que encuentre las ocurrencias de la frase *swimming vigorously*.**
<br>
Pista: Deberías incluir un patrón con `'IS_SPACE': True` entre las dos palabras.

In [11]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
pattern = [{'LOWER': 'swimming'}, {'IS_SPACE': True}, {'LOWER': 'vigorously'}]
matcher.add("Swimming", [pattern])


In [12]:
found_matches = matcher(doc)
found_matches




[(12881893835109366681, 1274, 1277), (12881893835109366681, 3609, 3612)]

**7. Imprime el texto al rededor de cada match encontrado**

In [13]:
start, end = found_matches[0][1:]
doc[start-9:end+13]

By diving I could evade the bullets and, swimming
vigorously, reach the bank, take to the woods and get away home

In [14]:
start, end = found_matches[1][1:]
doc[start-7:end+5]

over his shoulder; he was now swimming
vigorously with the current.  

**8. Imprime la oración que contiene cada match encontrado**

In [15]:
for sentence in sentences:
    for _, start, end in found_matches:
        if sentence.start <= start and sentence.end >= end:
            print(sentence.text, '\n')

By diving I could evade the bullets and, swimming
vigorously, reach the bank, take to the woods and get away home.   

The hunted man saw all this over his shoulder; he was now swimming
vigorously with the current.   

